# 🧠 CareerCompass AI Engine: Global Skill NER training (Autonomous)

This notebook trains a custom NER model to extract skills and roles. 

**Strategy:** We utilize a **Synthetic Data Augmentation** engine to bootstrap our training. This avoids dependencies on private/gated external datasets and ensures 100% reliability for the graduation project.

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate

## Step 2: Synthetic Data Engine
We generate high-quality annotated sentences using real-world skill/role dictionaries.

In [ ]:
import random
import json

SKILLS = ["Python", "Flutter", "Laravel", "React", "Node.js", "Docker", "Kubernetes", "SQL", "NoSQL", "AWS", "GCP", "Java", "C++", "Machine Learning", "Deep Learning", "NLP", "Firebase", "FastAPI", "Django", "Spring Boot", "Vue.js", "Tailwind CSS", "PostgreSQL", "MongoDB"]
ROLES = ["Backend Developer", "Frontend Developer", "Fullstack Developer", "Mobile Developer", "ML Engineer", "DevOps Engineer", "Data Scientist", "Project Manager", "UI/UX Designer", "Cybersecurity Analyst"]
TEMPLATES = [
    "Experienced in {skill1} and {skill2} as a {role}.",
    "Strong background in {skill1}, {skill2}, and {skill3}.",
    "Currently working as a {role} specializing in {skill1}.",
    "Proficient in {skill1}, {skill2}, and project management.",
    "Developed several applications using {skill1} and {skill2}.",
    "Passionate {role} with expertise in {skill1}.",
    "Hands-on experience with {skill1}, {skill2}, and {skill3}.",
    "Expertise in {skill1} in an enterprise environment.",
    "Implemented microservices using {skill1} and {skill2}.",
    "Senior {role} with 5 years of experience in {skill1}."
]

def generate_sample():
    template = random.choice(TEMPLATES)
    s1, s2, s3 = random.sample(SKILLS, 3)
    role = random.choice(ROLES)
    
    text = template.format(skill1=s1, skill2=s2, skill3=s3, role=role)
    tokens = text.replace(".", " .").replace(",", " ,").split()
    ner_tags = [0] * len(tokens)
    
    for i, token in enumerate(tokens):
        clean_token = token.strip(",.")
        if clean_token in s1 or clean_token in s2 or clean_token in s3:
            ner_tags[i] = 1 # SKILL
        elif clean_token in role:
            ner_tags[i] = 2 # ROLE
            
    return {"tokens": tokens, "ner_tags": ner_tags}

train_data = [generate_sample() for _ in range(5000)]
val_data = [generate_sample() for _ in range(1000)]

with open('train.json', 'w') as f:
    for entry in train_data: f.write(json.dumps(entry) + '\n')
with open('val.json', 'w') as f:
    for entry in val_data: f.write(json.dumps(entry) + '\n')

## Step 3: Load Data for Transformers

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
import evaluate

dataset = load_dataset('json', data_files={'train': 'train.json', 'test': 'val.json'})
label_list = ["O", "SKILL", "ROLE"]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

MODEL_CHECKPOINT = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None: label_ids.append(-100)
            elif word_idx != previous_word_idx: label_ids.append(label[word_idx])
            else: label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_ds = dataset.map(tokenize_and_align_labels, batched=True)

## Step 4: Training & Metrics

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=len(label_list), id2label=id2label, label2id=label2id)
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

args = TrainingArguments(
    "career-compass-ner", 
    eval_strategy="epoch", 
    save_strategy="epoch", 
    learning_rate=2e-5, 
    num_train_epochs=3, 
    weight_decay=0.01, 
    push_to_hub=False
)

# Note: In latest transformers (v4.46+), 'tokenizer' argument is renamed to 'processing_class'
trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=tokenized_ds["train"], 
    eval_dataset=tokenized_ds["test"], 
    processing_class=tokenizer, 
    data_collator=DataCollatorForTokenClassification(tokenizer), 
    compute_metrics=compute_metrics
)
trainer.train()

## Step 5: Save Model Weights

In [ ]:
trainer.save_model("career_compass_ner_final")
print("Training Complete! Download 'career_compass_ner_final' and add to your project.")